In [1]:
import pyterrier as pt
if not pt.started():
    pt.init()
    
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer
import numpy as np
# from pyterrier_splade import PyTerrierSPLADE
import pyterrier_dr
from ir_measures import *
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz
import json
from pyterrier.model import add_ranks
import os

/opt/miniconda3/envs/splade/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTerrier 0.10.1 has loaded Terrier 5.10 (built by craigm on 2024-08-22 17:33) and terrier-helper 0.0.8



In [2]:
ohsumed_docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

In [3]:
merged = pd.merge(qrels, ohsumed_docs, on = "docno")
merged = merged[["qid", "docno", "label", "sensitivity"]]

sensitive_docs = merged[merged['sensitivity'] == 1]
non_sensitive_docs = merged[merged['sensitivity'] == 0]

# List of all qids
all_qids = merged['qid'].unique()

# List of qid with at least one sensitive document
sensitive_qids = sensitive_docs['qid'].unique()

# List of qid without any sensitive documents
non_sensitive_qids = [qid for qid in all_qids if qid not in sensitive_qids]
# Dictionary to hold the result
result = []

# For each qid with sensitive documents, select a random sensitive document
for qid in sensitive_qids:
    docs = sensitive_docs[sensitive_docs['qid'] == qid]
    random_doc = docs.sample(n=1)  # Randomly select one document
    result.append([qid, random_doc.docno.iloc[0]])

# For each qid without sensitive documents, select a random sensitive document
if non_sensitive_qids:
    random_sensitive_doc = sensitive_docs.sample(n=1)
    for qid in non_sensitive_qids:
        result.append([qid, random_sensitive_doc.docno.iloc[0]])

qid_sens_docs = pd.merge(pd.DataFrame(result, columns = ["qid", "sensitive_docno"]), ohsumed_docs, left_on = "sensitive_docno", right_on = "docno").rename(columns = {"text" : "query"})

In [4]:
def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, ohsumed_docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

In [5]:
def filter_out(run):
    merged = pd.merge(run, ohsumed_docs, on=["docno"])
    merged = merged[merged["t5_sensitivity"] == 0]
    merged = merged.drop(columns = ["rank", "sensitivity"])
    merged = add_ranks(merged)
    return merged

perform_filter = pt.apply.generic(filter_out)

In [19]:
import pyterrier as pt
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer
import numpy as np
import pyterrier_dr
from ir_measures import *
import pandas as pd
from scipy.sparse import csr_matrix, save_npz, load_npz, vstack
import json
from pyterrier.model import add_ranks
import os
from splade_train.model import SPLADEModel

global sims

class PyTerrierSensitivitySPLADE(pt.Transformer):

    def __init__(self, relevance_model_name, sensitivity_model_name, index_location, relevance_index_name, sensitivity_index_name, verbose = True, batch_size = 4, num_results = 1000, alpha = 0.5):
        self.relevance_tokenizer = AutoTokenizer.from_pretrained(relevance_model_name, clean_up_tokenization_spaces = True)
        self.sensitivity_tokenizer = AutoTokenizer.from_pretrained(sensitivity_model_name, clean_up_tokenization_spaces = True)
        self.relevance_model = AutoModelForMaskedLM.from_pretrained(relevance_model_name)
        self.sensitivity_model = AutoModelForMaskedLM.from_pretrained(sensitivity_model_name)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.relevance_model.to(self.device)
        self.sensitivity_model.to(self.device)
        self.verbose = verbose
        self.batch_size = batch_size
        self.index_location = index_location
        self.num_results = num_results
        self.relevance_index_name = relevance_index_name
        self.sensitivity_index_name = sensitivity_index_name
        self.alpha = alpha

    def doc_encode(self, inp, relevance):
        
        if relevance:
            model = self.relevance_model
            tokenizer = self.relevance_tokenizer
        else:
            model = self.sensitivity_model
            tokenizer = self.sensitivity_tokenizer
        
        texts = inp.text.tolist()
        it = range(0, len(texts), self.batch_size)
        all_vecs = []

        if self.verbose:
            it = pt.tqdm(it, desc='Processing Batches', unit='batches')

        for start_idx in it:
            rng = slice(start_idx, start_idx + self.batch_size)
            batch_texts = texts[rng]

            with torch.no_grad():
                tokens = tokenizer(batch_texts, padding=True, truncation=True, return_attention_mask = True, return_tensors='pt')
                tokens = {k: v.to(self.device) for k, v in tokens.items()}
                output = model(**tokens)
                vecs = torch.max(torch.log(1 + torch.relu(output.logits)) * tokens["attention_mask"].unsqueeze(-1), dim=1)[0].squeeze().detach().cpu().numpy()
                tokens = {k: v.to("cpu") for k, v in tokens.items()}
                if len(vecs.shape) == 1: vecs = vecs[np.newaxis, :] # Handle if there is a single document in the last batch
                all_vecs.extend(list(vecs))
            
        inp["doc_vec"] = all_vecs
        return inp

    def query_encode(self, inp, relevance):
        if relevance:
            model = self.relevance_model
            tokenizer = self.relevance_tokenizer
        else:
            model = self.sensitivity_model
            tokenizer = self.sensitivity_tokenizer

        text = inp["query"].tolist()
        all_vecs = []
        
        with torch.no_grad():
            tokens = tokenizer(text, padding=True, truncation=True, return_attention_mask = True, return_tensors='pt')
            tokens = {k: v.to(self.device) for k, v in tokens.items()}
            output = model(**tokens)
            vecs = torch.max(torch.log(1 + torch.relu(output.logits)) * tokens["attention_mask"].unsqueeze(-1), dim=1)[0].squeeze().detach().cpu().numpy()
            if len(vecs.shape) == 1: vecs = vecs[np.newaxis, :] # Handle if there is a single document in the last batch
            all_vecs.extend(list(vecs))
            
        inp["query_vec"] = all_vecs
        return inp

    def index(self, inp, batch_size=100000):
        inp = pt.tqdm(inp, desc='indexing', unit='dvec', disable=False)  # Leave tqdm as an iterator
        docno_dict = {}
        batch_counter = 0
        batch_docs = []

        if not os.path.isdir(self.index_location):
            os.mkdir(self.index_location)


        for i, doc in enumerate(inp):
            batch_docs.append(doc)
    
            # When the batch size is reached, process and save the batch
            if len(batch_docs) == batch_size:
                self._process_and_save_batch(batch_docs, batch_counter, docno_dict)
                batch_docs = []  # Reset the batch
                batch_counter += 1
    
        # Process any remaining documents in the last batch
        if batch_docs:
            self._process_and_save_batch(batch_docs, batch_counter, docno_dict)
    
        # Save the combined docno dictionary
        with open(f"{self.index_location}/{self.relevance_index_name}.json", 'w') as f:
            json.dump(docno_dict, f)
    
        # Combine all batch files into one index file
        self._combine_batches(batch_counter + 1)  # +1 because batch_counter starts at 0

    def _process_and_save_batch(self, batch_docs, batch_counter, docno_dict):
        batch_df = pd.DataFrame(batch_docs)
        
        # Encode relevance document vectors for the batch
        batch_df = self.doc_encode(batch_df, relevance = True)
        doc_vecs = np.array(batch_df['doc_vec'].tolist())
        sparse_matrix = csr_matrix(doc_vecs)
        batch_filename = f"{self.index_location}/{self.relevance_index_name}_batch_{batch_counter}.npz"
        save_npz(batch_filename, sparse_matrix)

        batch_df = self.doc_encode(batch_df, relevance = False)
        doc_vecs = np.array(batch_df['doc_vec'].tolist())
        sparse_matrix = csr_matrix(doc_vecs)
        batch_filename = f"{self.index_location}/{self.sensitivity_index_name}_batch_{batch_counter}.npz"
        save_npz(batch_filename, sparse_matrix)
    
        # Update docno_dict with the new documents
        for i, doc in enumerate(batch_docs):
            docno_dict[len(docno_dict)] = doc["docno"]

    def _combine_batches(self, num_batches):
        # Relevance
        combined_matrix = None
        for batch_counter in range(num_batches):
            print(f"Processing batch : {batch_counter}")
            batch_filename = f"{self.index_location}/{self.relevance_index_name}_batch_{batch_counter}.npz"
            sparse_matrix = load_npz(batch_filename)
            
            if combined_matrix is None:
                combined_matrix = sparse_matrix
            else:
                combined_matrix = vstack([combined_matrix, sparse_matrix], format='csr')
            
            # Remove the batch file to save disk space
            os.remove(batch_filename)
    
        # Save the final combined index
        save_npz(f"{self.index_location}/{self.relevance_index_name}.npz", combined_matrix)

        # Sensitivity
        combined_matrix = None
        for batch_counter in range(num_batches):
            print(f"Processing batch : {batch_counter}")
            batch_filename = f"{self.index_location}/{self.sensitivity_index_name}_batch_{batch_counter}.npz"
            sparse_matrix = load_npz(batch_filename)
            
            if combined_matrix is None:
                combined_matrix = sparse_matrix
            else:
                combined_matrix = vstack([combined_matrix, sparse_matrix], format='csr')
            
            # Remove the batch file to save disk space
            os.remove(batch_filename)
    
        # Save the final combined index
        save_npz(f"{self.index_location}/{self.sensitivity_index_name}.npz", combined_matrix)
        
    def load_index(self):
        self.relevance_index = load_npz(f"{self.index_location}/{self.relevance_index_name}.npz")
        self.sensitivity_index = load_npz(f"{self.index_location}/{self.sensitivity_index_name}.npz")
        self.lookup = json.load(open(f"{self.index_location}/{self.relevance_index_name}.json"))

    def retrieve(self, inp):
        df_rows = []
        for query in inp.iterrows():
            qid = query[1].qid

            rel_sens_doc = qid_sens_docs[qid_sens_docs["qid"] == qid]
            rel_sens_doc = rel_sens_doc[["qid", "query"]]
            
            query_frame = pd.DataFrame(query[1].to_dict(), index = [0])
            relevance_query_vector = self.query_encode(query_frame, relevance = True)["query_vec"].iloc[0]
            relevance_query_vector = csr_matrix(relevance_query_vector).T

            sensitivity_query_vector = self.query_encode(rel_sens_doc, relevance = False)["query_vec"].iloc[0]
            sensitivity_query_vector = csr_matrix(sensitivity_query_vector).T
            
            relevance_similarity_scores = self.relevance_index.dot(relevance_query_vector).toarray().ravel()
            sensitivity_similarity_scores = self.sensitivity_index.dot(sensitivity_query_vector).toarray().ravel()
            
            # similarity_scores = ((1 - self.alpha) * relevance_similarity_scores) + (self.alpha * -sensitivity_similarity_scores)
            k = self.alpha  # e.g., top 20%

            # Calculate the number of elements to be set to -1000000
            num_elements = len(sensitivity_similarity_scores)
            num_to_set = int(num_elements * (k / 100))
            
            # Find the threshold value for the top k%
            threshold = np.partition(sensitivity_similarity_scores, -num_to_set)[-num_to_set]
            
            # Create the new array with the same shape as similarity_scores
            new_array = np.zeros_like(sensitivity_similarity_scores)
            
            # Set the top k% values to -1000000
            new_array[sensitivity_similarity_scores >= threshold] = -1000

            similarity_scores = relevance_similarity_scores + new_array

            top_indices = np.argsort(similarity_scores)[-self.num_results:][::-1]
            scores = similarity_scores[top_indices]

            
            for idx in top_indices:
                docno = self.lookup[str(idx)]
                score = similarity_scores[idx]
                df_rows.append({
                    "qid": query_frame["qid"].iloc[0],
                    "query": query_frame["query"].iloc[0],
                    "docno" : docno,
                    "score" : score
                })

        return add_ranks(pd.DataFrame(df_rows))
            
    def transform(self, inp):
        if "text" in inp.columns and "doc_vec" not in inp.columns:
            inp = self.doc_encode(inp)
            
        if "query" in inp.columns and "query_vec" not in inp.columns:
            self.load_index()
            inp = self.retrieve(inp)
            
        return inp

splade = PyTerrierSensitivitySPLADE(
    relevance_model_name = 'naver/splade-cocondenser-ensembledistil',
    sensitivity_model_name =  "./splade_train/ohsumed_splade_model_100_epoch/",
    verbose = True,
    index_location = "./indices/ohsumed_splade_sensitivity_10_epoch",
    relevance_index_name = "ohsumed_relevance",
    sensitivity_index_name = "ohsumed_sensitivity",
    alpha = 10
)
        
pt.io.write_results(splade(queries), "./runs/ohsumed_sensitivity_splade_v2")

pt.Experiment(
    [
        pt.io.read_results("./runs/cocondenser_ensemledistill_ohsumed"),
        pt.io.read_results("./runs/cocondenser_ensemledistill_ohsumed") >> perform_filter,
        pt.io.read_results("./runs/ohsumed_sensitivity_splade_v1"),
        pt.io.read_results("./runs/ohsumed_sensitivity_splade_v2"),
        pt.io.read_results("./runs/ohsumed_proportionate_random_sensitive_splade")
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = [
        "Relevance SPLADE",
        "Relevance SPLADE >> Filter",
        "Sensitivity-Aware SPLADE",
        "Sensitivity-Aware SPLADE",
        "Proportionate Random Sensitive SPLADE"
    ]
)

/tmp/ipykernel_28660/1992672345.py:257: DeprecationWarning: Coercion of a dataframe into a transformer is deprecated; use a pt.Transformer.from_df() instead
  pt.io.read_results("./runs/cocondenser_ensemledistill_ohsumed") >> perform_filter,


,name,nDCG@10,sens_docs@10
0,Relevance SPLADE,0.519776,1.367925
1,Relevance SPLADE >> Filter,0.516676,1.103774
2,Sensitivity-Aware SPLADE,0.092689,1.283019
3,Sensitivity-Aware SPLADE,0.339581,1.075472
4,Proportionate Random Sensitive SPLADE,0.002699,4.000000


In [7]:
pt.Experiment(
    [
        pt.io.read_results("./runs/cocondenser_ensemledistill_ohsumed"),
        pt.io.read_results("./runs/cocondenser_ensemledistill_ohsumed") >> perform_filter,
        pt.io.read_results("./runs/ohsumed_sensitivity_splade_v1"),
        pt.io.read_results("./runs/ohsumed_sensitivity_splade_v2")
    ],
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = [
        "Relevance SPLADE",
        "Relevance SPLADE >> Filter",
        "Sensitivity-Aware SPLADE",
        "Sensitivity-Aware SPLADE"
    ]
)

/tmp/ipykernel_28660/3727830111.py:4: DeprecationWarning: Coercion of a dataframe into a transformer is deprecated; use a pt.Transformer.from_df() instead
  pt.io.read_results("./runs/cocondenser_ensemledistill_ohsumed") >> perform_filter,


,name,nDCG@10,sens_docs@10
0,Relevance SPLADE,0.519776,1.367925
1,Relevance SPLADE >> Filter,0.516676,1.103774
2,Sensitivity-Aware SPLADE,0.092689,1.283019
3,Sensitivity-Aware SPLADE,0.339581,1.075472
